# P8b: Weighted Sums and Prediction

**FIT1061 Introduction to Artificial Intelligence — Week 8**

In previous weeks you classified data using probabilities (P6) and word counts (P7). This week you'll learn a different approach: **weighted sums**. Instead of multiplying probabilities, you'll combine numbers using weights — and the sign of the result tells you the class.

This is how Frank Rosenblatt's **Perceptron** worked in 1958, the machine the *New York Times* called a device that "learns by doing." The maths is simpler than Naive Bayes, but the idea is powerful: it's the building block of every neural network.

### What are features?

In machine learning, a **feature** is a measurable property of the thing you're classifying. In P7, the features were words (does the email contain "free"?). This week, the features are **numbers** — specifically, two coordinates $(x_1, x_2)$ that describe a point in 2D space. Think of it like plotting a student on a graph where $x_1$ is their exam score and $x_2$ is their assignment score — two numbers that together describe each student.

The classifier's job is to draw a line through this space that separates two groups. Which side of the line a point falls on determines its predicted class (+1 or -1). The **weights** control where that line goes.

> **Prerequisites:** Complete P8a (NumPy Basics) before starting this notebook.

> **This is a milestone task.** Book a tutor discussion after completing all parts.

> **P8.1 is personalised by your Monash student ID** — see the 'Your personalised instance' section below.


---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from p8_helpers import (HAND_TRACE_POINTS, HAND_TRACE_LABELS,
                        HAND_TRACE_WEIGHTS, HAND_TRACE_BIAS,
                        TRAIN_POINTS, TRAIN_LABELS,
                        TEST_POINTS, TEST_LABELS,
                        XOR_POINTS, XOR_LABELS,
                        plot_data, plot_boundary,
                        print_dataset_summary, print_confusion_matrix)
print("Setup OK.")

---

## Warm-up: the six-point classify from the whiteboard

In the lecture we traced a **weighted sum** on the six points below. Work through it yourself to
warm up, then apply YOUR personalised weights to the task's dataset in Part 0.

Stuck? This is the exact example we filled in on the board: revisit the slides or the workshop.

Weights $\mathbf{w} = [2, -1]$, bias $b = -1$, so the **score $= 2x_1 - x_2 - 1$**. The rule:
score $\geq 0$ predicts $+1$, otherwise $-1$. Fill in the score and prediction $\hat y$ by hand:

| pt | $(x_1, x_2)$ | $2x_1 - x_2 - 1$ | $\hat y$ | true |
|---|---|---|---|---|
| A | $(2, 1)$ | ? | ? | $+1$ |
| B | $(3, 2)$ | ? | ? | $+1$ |
| C | $(3, 4)$ | ? | ? | $+1$ |
| D | $(1, 3)$ | ? | ? | $-1$ |
| E | $(0, 2)$ | ? | ? | $-1$ |
| F | $(0, 1)$ | ? | ? | $-1$ |

1. How many did this line get correct?
2. The boundary is where the score is zero. Solve $2x_1 - x_2 - 1 = 0$ for $x_2$: what line is it?

*(These weights and points are the lecture's. Part 0 below uses YOUR personalised weights on the
task's own six points.)*


---

## Your personalised instance (P8.1)

Your hand-trace `(weights, bias)` are personalised by your Monash student ID. The 6-point dataset stays shared. Different students get different lines, and different accuracies.

**Set your Monash student ID below**, then run the next two cells. Do not edit `task_instances.py`.


In [ ]:
# Replace the placeholder with your Monash student ID (8 digits).
STUDENT_ID = "REPLACE_WITH_YOUR_ID"

assert STUDENT_ID.isdigit() and len(STUDENT_ID) == 8, (
    "Set STUDENT_ID to your 8-digit Monash student ID (digits only, in quotes)."
)


In [ ]:
# Integrity check + instance generation.
# DO NOT EDIT task_instances.py — your tutor regenerates from your ID
# and compares fingerprints. Modifications will be caught.

from task_instances import generate_instance
from task_instances_lib import fingerprint

instance = generate_instance(STUDENT_ID)
linclf = instance["linclf"]

print(f"Student ID:                {STUDENT_ID}")
print(f"task_instances.py SHA-256: {fingerprint('task_instances.py')}")
print()
print(linclf.describe())


---

## Part 0: Hand-Trace: The Weighted Sum (P8.1)

Before writing any code, work through this on paper. A **weighted sum** combines inputs with weights:

$$\mathbf{w} \cdot \mathbf{x} + b = w_1 x_1 + w_2 x_2 + b$$

The **decision rule** is simple: if the weighted sum is $\geq 0$, predict **+1**. Otherwise predict **-1**.

You will apply YOUR personalised weight vector $\mathbf{w}$ and bias $b$ (set above) to a shared 6-point dataset. Compute $\mathbf{w}\cdot\mathbf{x} + b$ for each point by hand, apply the threshold, and compare with the true labels.

> **Different students get different weights.** Your accuracy will differ from a peer's; that's expected and pedagogically intentional. The Part 2 exercise asks you to find better weights.


In [ ]:
# The hand-trace data — use this to check your paper work.
# Weights and bias are PERSONALISED to you (set above).
MY_W = linclf.weights      # YOUR personalised w = [w1, w2]
MY_B = linclf.bias         # YOUR personalised bias b

print("Hand-trace dataset:")
print(f"  YOUR weights: w = {MY_W}, bias b = {MY_B}")
print()
print(f"  {'Point':<6} {'x1':>4} {'x2':>4}   {'Actual':>6}")
print(f"  {'-'*30}")
names = ['A', 'B', 'C', 'D', 'E', 'F']
for name, (x1, x2), label in zip(names, HAND_TRACE_POINTS, HAND_TRACE_LABELS):
    print(f"  {name:<6} {int(x1):>4} {int(x2):>4}   {label:>+6d}")

print()
print("Compute by hand for each point:")
print("  w·x + b = w1*x1 + w2*x2 + b")
print("  prediction = +1 if w·x + b >= 0 else -1")


In [ ]:
# Visualise the hand-trace points before computing
plot_data(HAND_TRACE_POINTS, HAND_TRACE_LABELS, "Hand-Trace Data (6 points)")
print("Blue circles = class +1, red squares = class -1.")
print("Notice: the two groups are spatially separated.")
print("Your weighted sum will draw a LINE through this space to separate them.")

### Check your hand trace

After filling in the table on paper with YOUR weights, answer these questions:

1. **How many did you get correct?** Different (w, b) gives different accuracy; yours is unlikely to be 6/6. Note your `_/6`.
2. **Which points are misclassified?** Look at the visualisation. Where does YOUR boundary line cut the plane, and which points are on the wrong side?
3. **Which point had the largest (most positive) weighted sum?** Which had the most negative? What does that mean geometrically?
4. **Can you guess a better $(\mathbf{w}, b)$** that would correctly classify all 6 points? You'll explore this in Part 2.


---

### Articulation (P8.1: graded with the photo)

For each of the 6 points in `HAND_TRACE_POINTS`, compute $\mathbf{w}\cdot\mathbf{x} + b$ using YOUR weights and YOUR bias. Show your working on paper (substitute YOUR numbers, intermediate steps). Photograph it.

Then fill in the answer cell below.


In [ ]:
# Articulation answers — type your hand-computed values here.
# Reference YOUR weights and bias printed above.

answers = {
    "weights": None,            # [w1, w2]
    "bias": None,               # int
    "sums": None,               # list of 6 ints (weighted sums for points A..F)
    "predictions": None,        # list of 6 ints (+1 or -1)
    "correct": None,            # int 0..6
    "misclassified_indices": None,  # list of ints (0-indexed positions of wrong predictions)
}

# Brief explanation (2-3 sentences) — where does YOUR boundary line cut
# the plane, and which points fall on the wrong side? Reference YOUR numbers:
explanation = (
    "TYPE YOUR EXPLANATION HERE."
)

for k, v in answers.items():
    assert v is not None, f"answers['{k}'] is still None"
assert explanation.strip() != "TYPE YOUR EXPLANATION HERE.", "Write your explanation."
print("Filled in. Photograph hand trace + this cell for P8.1.")


### Where is the boundary line, exactly? (geometry of $\mathbf{w}\cdot\mathbf{x}+b=0$)

The decision boundary is the set of points where the weighted sum is exactly zero. In 2D that is the equation of a line. Derive the line in $y = mx + c$ form so you can plot it by hand.

Start from $w_1 x_1 + w_2 x_2 + b = 0$. Treat $x_1$ as the independent variable (the horizontal axis) and solve for $x_2$.

**Fill in the missing steps:**

$$w_1 x_1 + w_2 x_2 + b = 0$$

$$w_2 x_2 = ?$$

$$x_2 = ?$$

So the boundary line has slope $-w_1/w_2$ and intercept $-b/w_2$.

**Check on the hand-trace weights $\mathbf{w} = [2, -1]$, $b = -1$.** Plug into your derived formula:

- Slope: $-w_1/w_2 = -2 / -1 = 2$
- Intercept: $-b/w_2 = -(-1)/(-1) = -1$

So the boundary is $x_2 = 2 x_1 - 1$. Verify by hand that points $(1, 1)$ and $(0, -1)$ lie on this line, and confirm they are also on the boundary geometrically (the weighted sum should equal zero). When you reach the visualisation in Part 5, the plotted line should match this slope and intercept.

---

## Part 1: Explore the Dataset

Now let's work with a larger dataset. We have 30 training points and 10 test points in 2D, drawn from two clusters.

In [ ]:
print_dataset_summary(TRAIN_POINTS, TRAIN_LABELS, "Training set")
print()
print_dataset_summary(TEST_POINTS, TEST_LABELS, "Test set")
print()
print("First 5 training points:")
for i in range(5):
    x = TRAIN_POINTS[i]
    y = TRAIN_LABELS[i]
    print(f"  x = ({x[0]:5.2f}, {x[1]:5.2f}),  label = {y:+d}")
print("  ...")

# Visualise
plot_data(TRAIN_POINTS, TRAIN_LABELS, "Training Data")
plt.show()

---

## Part 2: Implement Weighted Sum and Predict (P8.1)

Now implement what you did by hand. You'll write three functions, each building on the last.

### 2a. `weighted_sum`: compute w·x + b

In [ ]:
def weighted_sum(x, w, b):
    """Compute the weighted sum: w·x + b.

    Args:
        x: A 1D numpy array of features (e.g., [x1, x2]).
        w: A 1D numpy array of weights (same length as x).
        b: A float (the bias).

    Returns:
        A float: the weighted sum w·x + b.
    """
    # --- YOUR CODE HERE ---
    # Compute the dot product of the weight vector w and the input x,
    # then add the bias b.


    # --- END YOUR CODE ---


# Test on the first hand-trace point using YOUR personalised (w, b).
# Compare with your hand calculation for point A.
import numpy as np
x_test = HAND_TRACE_POINTS[0]   # point A = (3, 1)
w = np.array(MY_W)              # YOUR personalised weights
b = MY_B                        # YOUR personalised bias

result = weighted_sum(x_test, w, b)
print(f"weighted_sum({x_test}, w={MY_W}, b={MY_B}) = {result}")
print(f"This should match the value you computed by hand for point A.")

**Check:** The result should equal the weighted sum you computed by hand for point A using YOUR `(w, b)`. If they differ, recheck your `weighted_sum` code.

### 2b. `predict`: make a decision


In [ ]:
def predict(x, w, b):
    """Predict the class of a single point.

    Decision rule: if w·x + b >= 0, return +1. Otherwise return -1.

    Args:
        x: A 1D numpy array of features.
        w: A 1D numpy array of weights.
        b: A float (the bias).

    Returns:
        +1 or -1.
    """
    # --- YOUR CODE HERE ---
    # Use your weighted_sum function, then apply the threshold.


    # --- END YOUR CODE ---


# Test on hand-trace points using YOUR personalised (w, b).
w = np.array(MY_W)
b = MY_B
print(f"Predictions on hand-trace data using YOUR weights w={MY_W}, b={MY_B}:")
names = ['A', 'B', 'C', 'D', 'E', 'F']
correct = 0
for name, x, actual in zip(names, HAND_TRACE_POINTS, HAND_TRACE_LABELS):
    pred = predict(x, w, b)
    match = "OK" if pred == actual else "miss"
    if pred == actual:
        correct += 1
    print(f"  {name}: predict={pred:+d}, actual={actual:+d}  {match}")
print(f"\nAccuracy on hand-trace: {correct}/{len(HAND_TRACE_LABELS)}")


**Check:** Both the predictions and the accuracy printed above should match what you computed by hand for YOUR `(w, b)` in Part 0. Mismatch ⇒ check your `weighted_sum` or threshold logic.

### 2c. `classify_all`: classify an entire dataset


In [ ]:
def classify_all(X, w, b):
    """Classify every point in a dataset.

    Args:
        X: A 2D numpy array of shape (n, 2) — one row per point.
        w: A 1D numpy array of weights.
        b: A float (the bias).

    Returns:
        A 1D numpy array of predictions (+1 or -1), one per row of X.
    """
    # --- YOUR CODE HERE ---
    # Loop through each row of X, call predict() on it,
    # and collect the results into a list.
    predictions = []


    # --- END YOUR CODE ---
    return np.array(predictions)


# Test on hand-trace data
preds = classify_all(HAND_TRACE_POINTS, HAND_TRACE_WEIGHTS, HAND_TRACE_BIAS)
accuracy = np.mean(preds == HAND_TRACE_LABELS)
print(f"Hand-trace accuracy: {accuracy:.0%}")
print(f"Expected: 100%")

---

## Part 3: Classify the Dataset

Now use your functions on the larger dataset with weights **w = [1, 1]** and bias **b = -3**.

In [ ]:
# Classify with w = [1, 1], b = -3
w = np.array([1, 1])
b = -3

train_preds = classify_all(TRAIN_POINTS, w, b)
train_accuracy = np.mean(train_preds == TRAIN_LABELS)
print(f"Training accuracy: {int(train_accuracy * len(TRAIN_LABELS))}/{len(TRAIN_LABELS)} = {train_accuracy:.0%}")

# Show a few predictions
print("\nSample predictions:")
for i in range(5):
    x = TRAIN_POINTS[i]
    score = weighted_sum(x, w, b)
    pred = train_preds[i]
    actual = TRAIN_LABELS[i]
    match = "✓" if pred == actual else "✗"
    print(f"  ({x[0]:5.2f}, {x[1]:5.2f})  score={score:+.2f}  pred={pred:+d}  actual={actual:+d}  {match}")
print("  ...")

In [ ]:
# Now on the test set
test_preds = classify_all(TEST_POINTS, w, b)
test_accuracy = np.mean(test_preds == TEST_LABELS)
print(f"Test accuracy: {int(test_accuracy * len(TEST_LABELS))}/{len(TEST_LABELS)} = {test_accuracy:.0%}")

---

## Part 4: How confident is each prediction? (Margin)

`predict` gives a class. But which test points were *barely* on the right side of the line, and which were comfortably far from it? The geometric answer is the **margin**: the perpendicular distance from a point to the decision boundary.

The signed margin is:

$$\text{margin}(\mathbf{x}) = \frac{\mathbf{w} \cdot \mathbf{x} + b}{\|\mathbf{w}\|}$$

The **sign** tells you which side of the boundary $\mathbf{x}$ is on (positive → predicted +1, negative → predicted -1).
The **magnitude** tells you how far it is from the boundary (large → confident, near zero → borderline).

> **Why divide by $\|\mathbf{w}\|$?** Without it, $\mathbf{w} \cdot \mathbf{x} + b$ measures the weighted sum in the units of $\mathbf{w}$. Scaling $\mathbf{w}$ by 10 would scale the weighted sum by 10 even though the boundary line is the same. Dividing by $\|\mathbf{w}\|$ removes that arbitrary scaling and gives the actual perpendicular distance.

In [ ]:
def compute_margin(x, w, b):
    """Return the signed perpendicular distance from x to the boundary w*x + b = 0.

    Args:
        x: 1D NumPy array of shape (2,) — a single point.
        w: 1D NumPy array of shape (2,) — weight vector.
        b: float — bias.

    Returns:
        The signed margin (a float). Positive → point is on the +1 side.
    """
    # --- YOUR CODE HERE ---
    # Use np.linalg.norm(w) for the L2 norm of the weight vector.
    # Use np.linalg.norm(w) for the L2 norm of the weight vector. (||w||)

    pass  # remove this line
    # --- END YOUR CODE ---


# Quick check on the hand-trace data with w = [2, -1], b = -1
# Hand computation for point A = (3, 1):
#   w·x + b = 2*3 + (-1)*1 + (-1) = 4
#   ||w||   = sqrt(4 + 1) = sqrt(5) ≈ 2.236
#   margin  = 4 / sqrt(5) ≈ 1.789
m_A = compute_margin(np.array([3, 1]),
                     HAND_TRACE_WEIGHTS, HAND_TRACE_BIAS)
print(f"Margin for point A (3, 1): {m_A:.4f}")
print(f"Expected:                  {4 / np.sqrt(5):.4f}")

In [ ]:
# Apply to the test set with w = [1, 1], b = -3 and sort by absolute margin.
w = np.array([1, 1])
b = -3

print(f"{'point':>16s}  {'margin':>8s}  {'|margin|':>9s}  pred  actual")
print("-" * 58)

# Compute margin for every test point, store with (margin, x, actual)
records = []
for x, actual in zip(TEST_POINTS, TEST_LABELS):
    m = compute_margin(x, w, b)
    pred = 1 if m >= 0 else -1
    records.append((m, x, pred, actual))

# Sort ascending by absolute margin so the borderline points come first
records.sort(key=lambda r: abs(r[0]))

for m, x, pred, actual in records:
    mark = "✓" if pred == actual else "✗"
    pt = f"({x[0]:+5.2f}, {x[1]:+5.2f})"
    print(f"  {pt:>14s}  {m:+8.3f}  {abs(m):>9.3f}    {pred:+d}     {actual:+d}  {mark}")

**P8.4: Margin reflection** (3-5 sentences).

Look at the sorted table above:

1. Which test point is the **most confident** classification, and which is the **most borderline**? (Use the magnitude.)
2. Were any wrong predictions also low-margin (borderline)? Or was the classifier confidently wrong somewhere?
3. If you were using this classifier to decide whether to **flag** an email for a human reviewer, which margin threshold (in absolute value) would you choose, and why?

_Your answer here._

> **Connection to W09.** Margin is a *confidence score*. In W09 we will discuss why confidence scores from a classifier are often *not* well-calibrated probabilities, and what that means when the classifier is used to make decisions about people.

---

## Part 4b: Precision and recall

Accuracy alone is a blunt metric. It does not tell you *what kind* of mistakes the classifier is making. The same accuracy can hide very different behaviour:

- A classifier that calls everything +1 might be 70% accurate on a 70/30 split, but it never catches a -1.
- A classifier that is conservative (almost always predicts -1) might never make a false positive, but it also catches very few real +1s.

To separate these, we use **precision** and **recall**, computed from a **confusion matrix**.

|                           | predicted +1     | predicted -1     |
|---------------------------|------------------|------------------|
| **actual +1 (positive)**  | True positive (TP) | False negative (FN) |
| **actual -1 (negative)**  | False positive (FP) | True negative (TN)  |

You wrote almost identical code in P7; reuse the pattern.

In [ ]:
def count_confusion(predictions, labels):
    """Count (tp, fp, fn, tn) given prediction and label arrays.

    Convention: positive class is +1, negative class is -1.

    Args:
        predictions: NumPy array of +1/-1 predictions, shape (n,).
        labels:      NumPy array of +1/-1 actual labels, shape (n,).

    Returns:
        Tuple (tp, fp, fn, tn) of Python ints.
    """
    # --- YOUR CODE HERE ---
    # Count each combination by comparing element-wise.
    # For TP, count the cases where the prediction is 1 and the true label is also 1.
    # Cast the np.int64 result to int before returning each count.

    pass  # remove this line
    # --- END YOUR CODE ---


# Apply to the test set with w = [1, 1], b = -3
test_preds = np.array([1 if compute_margin(x, w, b) >= 0 else -1
                       for x in TEST_POINTS])

tp, fp, fn, tn = count_confusion(test_preds, TEST_LABELS)
print_confusion_matrix(tp, fp, fn, tn)

The two metrics are:

$$\text{precision} = \frac{\text{TP}}{\text{TP} + \text{FP}} \qquad \text{recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}$$

- **Precision.** Of the points the model said are +1, what fraction actually are?
- **Recall.** Of the points that actually are +1, what fraction did the model catch?

Implement them below.

In [ ]:
def precision(tp, fp):
    """Return precision = TP / (TP + FP). Return 0.0 if TP + FP == 0."""
    # --- YOUR CODE HERE ---


    # --- END YOUR CODE ---


def recall(tp, fn):
    """Return recall = TP / (TP + FN). Return 0.0 if TP + FN == 0."""
    # --- YOUR CODE HERE ---


    # --- END YOUR CODE ---


prec = precision(tp, fp)
rec = recall(tp, fn)
print(f"Precision: {prec:.2%}")
print(f"Recall:    {rec:.2%}")

**P8.4b: Precision vs recall reflection** (3-5 sentences).

1. Compare your test accuracy from Part 3 with the precision and recall above. Do they tell the same story, or are they hiding different errors?
2. If this classifier were deciding which loan applications to flag as "high risk" (positive class), what would a **false positive** cost? What would a **false negative** cost? Which metric (precision or recall) matters more, and why?

_Your answer here._

> **Connection to W09 (fairness).** When the positive and negative classes correspond to people, precision and recall mean very different things for different groups. We will return to this in W09 with `subgroup` precision/recall comparisons on a real dataset.

---

## Part 5: Visualise Decision Boundaries (P8.2)

The **decision boundary** is the line where $\mathbf{w} \cdot \mathbf{x} + b = 0$. Points on one side get classified as +1, points on the other as -1. Let's see it.

In [ ]:
# Visualise the boundary for w = [1, 1], b = -3
plot_boundary(TRAIN_POINTS, TRAIN_LABELS,
              w=np.array([1, 1]), b=-3,
              title="w = [1, 1], b = -3")

### Experiment with different weights

Now try **three different weight vectors** of your own. For each one:
1. Set `w` and `b` below.
2. Run the cell to see the boundary and accuracy.
3. Record the accuracy.

**Goal:** Find the best weight vector you can, and also try one that performs badly. Observe how the weights change the boundary.

**Hints:**
- Changing `w[0]` relative to `w[1]` **rotates** the line.
- Changing `b` **shifts** the line closer or further from the origin.
- Try `w = [1, 0], b = 0`: what happens when one weight is zero?

In [ ]:
# --- Experiment 1 ---
# Try your own weights! Replace the values below.
w1 = np.array([1, 0])   # ← change these
b1 = 0                   # ← change this

plot_boundary(TRAIN_POINTS, TRAIN_LABELS, w=w1, b=b1,
              title=f"Experiment 1: w = {w1.tolist()}, b = {b1}")

In [ ]:
# --- Experiment 2 ---
w2 = np.array([1, -1])  # ← change these
b2 = 0                   # ← change this

plot_boundary(TRAIN_POINTS, TRAIN_LABELS, w=w2, b=b2,
              title=f"Experiment 2: w = {w2.tolist()}, b = {b2}")

In [ ]:
# --- Experiment 3 ---
w3 = np.array([2, 3])   # ← change these
b3 = -5                  # ← change this

plot_boundary(TRAIN_POINTS, TRAIN_LABELS, w=w3, b=b3,
              title=f"Experiment 3: w = {w3.tolist()}, b = {b3}")

### Boundary analysis

Answer these questions (write your answers here or on paper):

1. **Which weight vector gave the highest accuracy?** Why do you think it works well?
2. **Which gave the lowest accuracy?** What went wrong geometrically?
3. **How did changing `b` (the bias) affect the boundary?** Try keeping `w` the same but changing `b`: what do you see?
4. **How did changing the ratio of `w[0]` to `w[1]` affect the boundary angle?**

> **Key insight:** Finding the right weights is the whole game. In Week 8's lecture, you'll see how the perceptron *learns* good weights automatically. For now, you're doing it by hand, like tuning a radio dial.

---

## Part 6: When No Line Works: The XOR Problem

Not every dataset can be separated by a line. Try to classify the **XOR** (exclusive or) data:

| x₁ | x₂ | Label |
|----|----|-------|
| 0  | 0  | -1    |
| 0  | 1  | +1    |
| 1  | 0  | +1    |
| 1  | 1  | -1    |

Opposite corners share a label. Can you find weights that get all 4 correct?

In [ ]:
# Try to classify XOR — change w and b to get all 4 correct
w_xor = np.array([1, 1])  # ← try different values
b_xor = -0.5              # ← try different values

plot_boundary(XOR_POINTS, XOR_LABELS, w=w_xor, b=b_xor,
              title=f"XOR: w = {w_xor.tolist()}, b = {b_xor}")

### What went wrong?

No matter what weights you try, you can't get all 4 XOR points correct. The best any line can do is **3 out of 4**.

**Why?** The +1 points (top-left and bottom-right corners) are on *opposite* sides of the square. No single straight line can separate them from the -1 points. This is called **linear inseparability**.

In 1969, Marvin Minsky and Seymour Papert published a book called *Perceptrons* that proved this mathematically. Their proof showed that a single-layer perceptron could never solve XOR, and by extension, many other useful problems.

The reaction was devastating:
- **Research funding collapsed.** Agencies that had poured money into neural networks pulled out.
- **Careers were destroyed.** Researchers who had bet on neural networks found themselves unemployable.
- **The field went dormant for nearly 20 years** (a period now called the **AI Winter**, roughly 1969–1986).

It wasn't until 1986, when Rumelhart, Hinton, and Williams showed how to train *multi-layer* networks (using backpropagation), that neural networks came back to life.

> **The limitation is real, but the conclusion was wrong.** One perceptron can't solve XOR. But *two perceptrons feeding into a third* can. Minsky and Papert knew this, but training multi-layer networks seemed impossible at the time. The lesson: don't dismiss a tool because of one limitation.

In [ ]:
from jupyterquiz import display_quiz

weighted_sums_quiz = [
    {
        "question": "A point has features x = (2, 5) and the weights are w = (3, -1) with bias b = 0. What is the weighted sum, and what class is predicted?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "Weighted sum = 1, predict +1", "correct": True, "feedback": "Correct! w·x + b = 3×2 + (-1)×5 + 0 = 6 - 5 = 1. Since 1 ≥ 0, predict +1."},
            {"answer": "Weighted sum = 1, predict -1", "correct": False, "feedback": "The weighted sum is right (3×2 + (-1)×5 = 1), but 1 ≥ 0, so the prediction should be +1."},
            {"answer": "Weighted sum = 11, predict +1", "correct": False, "feedback": "Check the signs: w₂ = -1, so the second term is (-1)×5 = -5, not +5."},
            {"answer": "Weighted sum = -1, predict -1", "correct": False, "feedback": "Recompute: 3×2 = 6, (-1)×5 = -5, and b = 0. So 6 + (-5) + 0 = 1, not -1."}
        ]
    },
    {
        "question": "You have a weight vector w = (1, 1) with bias b = -3. What does the <b>bias</b> do to the decision boundary?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "It shifts the boundary line away from the origin — without it, the boundary would always pass through (0, 0).", "correct": True, "feedback": "Correct! The boundary is the line w·x + b = 0. With b = 0, that line passes through the origin. The bias lets you move it."},
            {"answer": "It rotates the boundary line.", "correct": False, "feedback": "The angle of the boundary is controlled by the ratio of the weights (w₁/w₂), not by b. The bias shifts the line without changing its angle."},
            {"answer": "It scales the weights to make them larger.", "correct": False, "feedback": "The bias is added after the dot product — it doesn't change the weights. It shifts the threshold for the decision."},
            {"answer": "It has no effect on the boundary.", "correct": False, "feedback": "Try plotting the same weights with b = 0 and b = -3. The boundary moves! That's the bias at work."}
        ]
    },
    {
        "question": "Why can't a single weighted sum (with any weights) correctly classify the XOR data?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "Because a weighted sum draws a straight line, and the +1 points are on opposite corners — no straight line can put them both on the same side.", "correct": True, "feedback": "Correct! This is linear inseparability. A single line divides the plane into two halves, but the +1 points (0,1) and (1,0) are in different halves for any line."},
            {"answer": "Because the weights aren't large enough to separate the points.", "correct": False, "feedback": "Scaling up the weights doesn't help — it changes the magnitude of the scores but not which side of the boundary each point falls on."},
            {"answer": "Because XOR has too few data points to learn from.", "correct": False, "feedback": "Four points is enough — BFS in P2 worked with small inputs. The issue is geometric: no line can separate opposite corners of a square."},
            {"answer": "Because the bias is always wrong for XOR.", "correct": False, "feedback": "No value of b fixes it. The problem is fundamental: a straight line (any w, any b) cannot separate opposite corners."}
        ]
    }
]

display_quiz(weighted_sums_quiz, shuffle_answers=True)

---

## Part 7: Record Your Results

Fill in the table below with your results:

| Metric | Value |
|--------|-------|
| Hand-trace accuracy (YOUR personalised w, b) | ___/6 |
| Training accuracy (w=[1,1], b=-3) | ___% |
| Test accuracy (w=[1,1], b=-3) | ___% |
| Best weight vector you found | w = [___, ___], b = ___ |
| Best training accuracy | ___% |
| XOR: best you achieved | ___/4 |

---

### P8.3: Reflection: "What lessons should today's AI hype cycle learn from the AI Winter?"

Write a short response (150–250 words). Consider:

- The perceptron was announced as a thinking machine. What was the gap between the hype and reality?
- Minsky and Papert's proof killed funding for a promising idea. Was the reaction proportionate?
- Today's AI systems (ChatGPT, self-driving cars) are announced with similar enthusiasm. What patterns repeat?
- What would a responsible approach to AI expectations look like?

**Write your response in the cell below — it stays in this notebook.** You do not upload it as a separate file.

---

**Your response:**

*(Write here)*

---

## Submission Checklist

- [ ] **P8.1:** Hand computation of weighted sums for 6 points (photographed) — this is your single appended file
- [ ] **P8.1:** `weighted_sum`, `predict`, `classify_all` implemented and working
- [ ] **P8.1:** Training and test accuracy computed
- [ ] **Part 0:** Boundary line derivation filled in (slope and intercept from $\mathbf{w}\cdot\mathbf{x}+b=0$)
- [ ] **Part 4:** `compute_margin` implemented; sorted-margin table for test set; reflection answered
- [ ] **Part 4b:** `count_confusion`, `precision`, `recall` implemented; confusion matrix printed; reflection answered
- [ ] **Part 5:** Decision boundaries visualised for 3 different weight vectors
- [ ] **Part 5:** Boundary analysis questions answered
- [ ] **Part 6:** XOR attempt and "what went wrong" reflection written
- [ ] **Part 7:** Results table filled in + AI Winter reflection (150-250 words, written in this notebook)
- [ ] Milestone discussion booked with tutor

**Uploading to OnTrack.** Submit **two things**: (1) this completed notebook, and (2) a **single** photo or PDF of your hand computation (P8.1). Your written reflections stay *inside this notebook* — OnTrack allows only one extra file besides the notebook, and that file is your hand computation.